In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Data Preparation

In [3]:
anime_df = pd.read_csv(r'C:\Users\attaj\Downloads\anime.csv')
anime_df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [5]:
rating_df = pd.read_csv(r'C:\Users\attaj\Downloads\rating.csv')
rating_df.head()

,user_id,anime_id,rating
0,1,20,-1
1,1,24,-1
2,1,79,-1
3,1,226,-1
4,1,241,-1


## Merge Dataset

In [7]:
# Satukan rating dan metadata anime
anime = anime_df.copy()
rating = rating_df.copy()

# Gabungkan kedua dataset berdasarkan kolom 'anime_id'
merged_df = pd.merge(rating, anime, on='anime_id')

# Lihat hasilnya
print("=== Dataset Gabungan ===")
display(merged_df.head())

# Cek ukuran dataset gabungan
print(f"\nJumlah baris: {merged_df.shape[0]}")
print(f"Jumlah kolom: {merged_df.shape[1]}")

# Simpan hasil gabungan ke file di lokasi yang sama
output_path = r"C:\Users\attaj\Downloads\anime_rating_merged.csv"
merged_df.to_csv(output_path, index=False)
print(f"\nFile hasil gabungan telah disimpan sebagai '{output_path}'")

=== Dataset Gabungan ===


,user_id,anime_id,rating_x,name,genre,type,episodes,rating_y,members
0,1,20,-1,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,220,7.81,683297
1,1,24,-1,School Rumble,"Comedy, Romance, School, Shounen",TV,26,8.06,178553
2,1,79,-1,Shuffle!,"Comedy, Drama, Ecchi, Fantasy, Harem, Magic, R...",TV,24,7.31,158772
3,1,226,-1,Elfen Lied,"Action, Drama, Horror, Psychological, Romance,...",TV,13,7.85,623511
4,1,241,-1,Girls Bravo: First Season,"Comedy, Ecchi, Fantasy, Harem, Romance, School",TV,11,6.69,84395



Jumlah baris: 7813727
Jumlah kolom: 9

File hasil gabungan telah disimpan sebagai 'C:\Users\attaj\Downloads\anime_rating_merged.csv'


In [8]:
# Bersihkan rating dari dataset gabungan untuk collaborative filtering
cf_ratings = merged_df[["user_id", "anime_id", "name", "rating_x"]].copy()
cf_ratings = cf_ratings[cf_ratings["rating_x"] != -1]
cf_ratings["rating_x"] = pd.to_numeric(cf_ratings["rating_x"], errors="coerce")
cf_ratings = cf_ratings.dropna(subset=["rating_x"])
cf_ratings = cf_ratings.rename(columns={"rating_x": "rating"})
cf_ratings.head()

,user_id,anime_id,name,rating
47,1,8074,Highschool of the Dead,10
81,1,11617,High School DxD,10
83,1,11757,Sword Art Online,10
101,1,15451,High School DxD New,10
153,2,11771,Kuroko no Basket,10


## Split Data

In [9]:
# Pisahkan data menjadi train dan test
train_df, test_df = train_test_split(cf_ratings, test_size=0.2, random_state=42)
test_df = test_df[test_df["user_id"].isin(train_df["user_id"])]
print(f"Train interactions: {len(train_df)}")
print(f"Test interactions: {len(test_df)}")

Train interactions: 5069791
Test interactions: 1266528


## Build Model CF

In [10]:
# Bentuk matriks user-item berbasis data train dan hitung kemiripan
max_users = 5000  # batasi jumlah user agar perhitungan similarity tidak boros memori
if train_df["user_id"].nunique() > max_users:
    active_users = train_df["user_id"].value_counts().head(max_users).index
    train_subset = train_df[train_df["user_id"].isin(active_users)]
else:
    train_subset = train_df.copy()

user_anime_matrix = train_subset.pivot_table(
    index="user_id",
    columns="anime_id",
    values="rating"
 )
user_anime_filled = user_anime_matrix.fillna(0)
user_similarity = pd.DataFrame(
    cosine_similarity(user_anime_filled),
    index=user_anime_filled.index,
    columns=user_anime_filled.index
)
print(f"Jumlah user dalam matriks: {user_anime_matrix.shape[0]}")
print(f"Jumlah anime dalam matriks: {user_anime_matrix.shape[1]}")

Jumlah user dalam matriks: 5000
Jumlah anime dalam matriks: 9537


## Predict Ratings

In [11]:
# Fungsi untuk memprediksi rating seorang user terhadap anime tertentu
def predict_rating(user_id: int, anime_id: int, top_n_neighbors: int = 20):
    if user_id not in user_anime_matrix.index or anime_id not in user_anime_matrix.columns:
        return np.nan

    user_sim = user_similarity.loc[user_id].drop(user_id)
    neighbor_ratings = user_anime_matrix[anime_id].dropna()
    common_neighbors = neighbor_ratings.index.intersection(user_sim.index)
    if common_neighbors.empty:
        return np.nan

    top_neighbors = user_sim.loc[common_neighbors].nlargest(top_n_neighbors)
    neighbor_values = user_anime_matrix.loc[top_neighbors.index, anime_id]
    weights = top_neighbors.clip(lower=0)
    weight_sum = weights.sum()
    if weight_sum == 0:
        return np.nan
    return np.dot(neighbor_values, weights) / weight_sum

## Evaluate Model

In [13]:
# Evaluasi dengan menghitung MAE dan RMSE pada data test
eval_df = test_df[
    test_df["user_id"].isin(user_anime_matrix.index) &
    test_df["anime_id"].isin(user_anime_matrix.columns)
].copy()
eval_df["pred_rating"] = eval_df.apply(
    lambda row: predict_rating(row["user_id"], row["anime_id"]), axis=1
 )
eval_df = eval_df.dropna(subset=["pred_rating"])

if not eval_df.empty:
    mse = mean_squared_error(eval_df["rating"], eval_df["pred_rating"])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(eval_df["rating"], eval_df["pred_rating"])
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE : {mae:.4f}")
else:
    print("Tidak ada pasangan user-item di data test yang bisa diprediksi.")

RMSE: 1.3815
MAE : 1.0630


## Generate Recommendation

In [14]:
# Buat rekomendasi anime untuk user tertentu
def get_user_recommendations(target_user_id: int, top_n_similar: int = 10, top_n_recs: int = 10):
    if target_user_id not in user_similarity.index:
        raise ValueError("User tidak terdapat pada subset matriks (atur ulang max_users atau pilih user lain).")

    similar_users = user_similarity.loc[target_user_id].drop(target_user_id).nlargest(top_n_similar)
    if similar_users.empty:
        raise ValueError("Tidak ada user serupa yang ditemukan. Coba ubah parameter atau tambah data.")

    weighted_scores = user_anime_filled.loc[similar_users.index].T.dot(similar_users)
    already_watched = user_anime_matrix.loc[target_user_id].dropna().index
    recommendations = weighted_scores.drop(already_watched, errors="ignore").sort_values(ascending=False)

    rec_df = recommendations.head(top_n_recs).reset_index()
    rec_df.columns = ["anime_id", "score"]
    metadata_cols = ["anime_id", "name", "genre", "type"]
    rec_df = rec_df.merge(anime[metadata_cols].drop_duplicates("anime_id"), on="anime_id", how="left")
    return rec_df

example_user = train_subset["user_id"].iloc[0]
print(f"Contoh rekomendasi untuk user {example_user}:")
display(get_user_recommendations(example_user))

Contoh rekomendasi untuk user 60178:


,anime_id,score,name,genre,type
0,16498,33.438384,Shingeki no Kyojin,"Action, Drama, Fantasy, Shounen, Super Power",TV
1,1575,32.404551,Code Geass: Hangyaku no Lelouch,"Action, Mecha, Military, School, Sci-Fi, Super...",TV
2,7674,30.353210,Bakuman.,"Comedy, Romance, Shounen",TV
3,3783,30.010433,Kara no Kyoukai 3: Tsuukaku Zanryuu,"Action, Drama, Mystery, Supernatural, Thriller",Movie
4,2966,29.231707,Ookami to Koushinryou,"Adventure, Fantasy, Historical, Romance",TV
5,4282,28.442939,Kara no Kyoukai 5: Mujun Rasen,"Action, Drama, Mystery, Romance, Supernatural,...",Movie
6,4280,28.151557,Kara no Kyoukai 4: Garan no Dou,"Action, Mystery, Supernatural, Thriller",Movie
7,13125,27.072720,Shinsekai yori,"Drama, Horror, Mystery, Sci-Fi, Supernatural",TV
8,164,26.700074,Mononoke Hime,"Action, Adventure, Fantasy",Movie
9,7338,25.211449,Darker than Black: Kuro no Keiyakusha Gaiden,"Action, Mystery, Sci-Fi, Super Power",Special
